In [9]:
#setup installing package to the venv
%pip install pandas pyarrow pyspark
%pip install install-jdk

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [13]:
#java home setup
%pip install install-jdk

import os
import sys
import jdk

# 1. Target the jvm directory inside your active virtual environment
venv_path = sys.prefix
jvm_dir = os.path.join(venv_path, "jvm")

# 2. Download and extract OpenJDK 17 directly into .venv/jvm
java_home = jdk.install('17', path=jvm_dir)

# 3. Expose JAVA_HOME and bin to the active process
os.environ["JAVA_HOME"] = java_home
os.environ["PATH"] = os.path.join(java_home, "bin") + ":" + os.environ["PATH"]

print(f"JAVA_HOME successfully set to: {os.environ['JAVA_HOME']}")

Note: you may need to restart the kernel to use updated packages.
JAVA_HOME successfully set to: /home/wnder/Documents/repos/teleSpikeRedo/.venv/jvm/jdk-17.0.20+8


In [14]:
import re

import sqlite3
import pandas as pd

from pyspark.sql import SparkSession
from pyspark.sql import functions as SqlFun
from pyspark.sql.types import ArrayType, StringType


#def GetBaseline(token):

# a simple worker functaion for parsing one message into tokes
def tokenize(text):
    if not text:
        return []
    #Hebrew and alphanumeric words of length >= 2
    return re.findall(r"[\u0590-\u05fe\w]{2,}", text.lower())

#conver time stamp into hour and date
def add_time_columns(df):
    return df.withColumn("dt", SqlFun.to_timestamp(SqlFun.col("ts"))) \
             .withColumn("hour", SqlFun.hour(SqlFun.col("dt"))) \
             .withColumn("date", SqlFun.to_date(SqlFun.col("dt")))

#tokenise and exploead 
def explode_and_filter_tokens(df, allowed_tokens=None):
    tokenize_udf = SqlFun.udf(tokenize, ArrayType(StringType()))
    
    tokens_df = df.withColumn("word", SqlFun.explode(tokenize_udf(SqlFun.col("text"))))
    
    if allowed_tokens:
        tokens_df = tokens_df.filter(SqlFun.col("word").isin(allowed_tokens))
        
    return tokens_df

# 4. Aggregation & 24-Hour Pivot (Distributed in Spark)
def compute_hourly_pivots(tokens_df, total_days):
    # group rows per word and hour
    counts = tokens_df.groupBy("word", "hour").count()
    
    # Pivot hours into columns (now each word has a clolumb for each time of day)
    pivoted = counts.groupBy("word").pivot("hour", list(range(24))).sum("count").na.fill(0)
    
    # divide counts by total days to get average baseline per hour
    # Rename columns to h0, h1 ... h23
    for h in range(24):
        pivoted = pivoted.withColumn(f"h{h}", SqlFun.col(str(h)) / total_days).drop(str(h))
        
    return pivoted

#main function calling other parts    
def generate_baseline_table(sqlite_path, output_table_path, allowed_tokens):
    #this will recive an sql full of thounsds or millions of messages and a 
    #list of importent tokens. it will clean and tokenise each message, filter not importent tokens like "and" "if". any thing that isnt in the list.
    #it will then save in a small sql table
    #a row for each token with a columb for each hour of the day, and save the avrage apperenses in that hour for each token. we can than devide by 60 or 240 to get baselines for our time window

    spark = SparkSession.builder \
        .appName("BaselineGenerator") \
        .config("spark.driver.memory", "2g") \
        .getOrCreate()

    # load messages from local data base into Spark
    conn = sqlite3.connect(sqlite_path)
    pdf = pd.read_sql_query("SELECT text, ts FROM messages", conn)
    conn.close()

    if pdf.empty:
        print("No messages found.")
        return

    raw_df = spark.createDataFrame(pdf)

    #add time of day columb to data frame, parsed from the timestamp that came with the message
    #as well as a date columb for later calcualtions
    timed_df = add_time_columns(raw_df)

    # Calculate total days for avrge calculations later on
    total_days = max(1, timed_df.select("date").distinct().count())

    # tokenization and explode into rows
    tokens_df = explode_and_filter_tokens(timed_df, allowed_tokens=allowed_tokens)

    # use the data to turn the table into each row has: word h0 h1...., in each cloumb have the avrage for that time of day
    baseline_matrix = compute_hourly_pivots(tokens_df, total_days)

    # save to sql
    baseline_pdf = baseline_matrix.toPandas()
    
    conn_out = sqlite3.connect(output_table_path)
    baseline_pdf.to_sql("token_baselines", conn_out, if_exists="replace", index=False)
    conn_out.close()
    
    print(f"Generated baselines for {len(baseline_pdf)} tokens over {total_days} days.")
    return baseline_pdf

In [16]:
# Run the pipeline on your scraped messages
baselines_df = generate_baseline_table(
    sqlite_path="messages.db",
    output_table_path="baselines.db",
    allowed_tokens=["טיל","ישראל"]  # Set to a list like ["טיל", "אזעקה"] if you want to filter, or None for all
)

# Preview the top tokens at 14:00 (2:00 PM)
if baselines_df is not None:
    print(baselines_df[["word", "h14"]].sort_values(by="h14", ascending=False).head(10))

/home/wnder/Documents/repos/teleSpikeRedo/.venv/lib/python3.13/site-packages/pyspark/sql/pandas/conversion.py:659: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
/home/wnder/Documents/repos/teleSpikeRedo/.venv/lib/python3.13/site-packages/pyspark/sql/pandas/conversion.py:936: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
/home/wnder/Documents/repos/teleSpikeRedo/.venv/lib/python3.13/site-packages/pyspark/sql/udf.py:116: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
/home/wnder/Documents/repos/teleSpikeRedo/.venv/lib/python3.13/site-packages/pyspark/sql/pandas/conver

Generated baselines for 2 tokens over 192 days.
    word       h14
0  ישראל  0.500000
1    טיל  0.010417


In [ ]:
#print data base messages.db
conn = sqlite3.connect("messages.db")
df_msgs = pd.read_sql_query("SELECT * FROM messages LIMIT 20", conn)
conn.close()
print("Messages Table:")
display(df_msgs)

Messages Table:


,channel,sender,text,ts
0,https://t.me/koahadasotbatelegram,-1001254976833,הילד בן ה-4 אשר נפצע באירוע הדקירה בבית שמש בת...,1787216895
1,https://t.me/koahadasotbatelegram,-1001254976833,"לשכת ראש הממשלה: בניגוד לדיווחים, הדרג המדיני ...",1787216204
2,https://t.me/koahadasotbatelegram,-1001254976833,עוקבים יקרים שלנו! ❤️\n\nרוצים להישאר מעודכנים...,1787214544
3,https://t.me/koahadasotbatelegram,-1001254976833,**תוכן שיווקי:**,1787214499
4,https://t.me/koahadasotbatelegram,-1001254976833,הבוקר הפך למרגש במיוחד עבור לוחמי האש מתחנת נו...,1787212462
5,https://t.me/koahadasotbatelegram,-1001254976833,בשורה לנוסעים לקולומביה: בעקבות שיחות בין שר ה...,1787208197
6,https://t.me/koahadasotbatelegram,-1001254976833,חיבור חדש בימין: מפלגות צומת ובית ישראל הודיעו...,1787206021
7,https://t.me/koahadasotbatelegram,-1001254976833,תחזית מזג האוויר: לאחר התפזרות עננות הבוקר יעש...,1787200315
8,https://t.me/koahadasotbatelegram,-1001254976833,דיווחים על תקיפות בלבנון במהלך הלילה\n\nצילום:...,1787199251
9,https://t.me/koahadasotbatelegram,-1001254976833,טראמפ בהודעה דרמטית: איראן קיבלה הזדמנות להגיע...,1787198454


In [22]:
#print basline.db
conn = sqlite3.connect("baselines.db")
df_base = pd.read_sql_query("SELECT word, h0, h8, h14, h20 FROM token_baselines ORDER BY h14 DESC LIMIT 10", conn)
conn.close()
print("Baselines Table:")
display(df_base)

Baselines Table:


,word,h0,h8,h14,h20
0,ישראל,0.192708,0.255208,0.500000,0.666667
1,טיל,0.041667,0.015625,0.010417,0.072917
